# FEM Plate-with-Hole GNN Surrogate — Colab GPU setup

Sibling notebook to the AirfRANS project's `colab_setup.ipynb`, simplified: this
dataset (200 Abaqus cases, ~5-6k nodes each, ~207MB raw + ~78MB cached) is small
enough to just live in the git repo itself -- unlike AirfRANS's ~15GB dataset,
there's no external download step, no Drive mount needed for the data. All real
logic still lives in `src/`; this notebook is infrastructure only.

Runtime > Change runtime type > GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Colab VMs reset every session, so the repo needs to come from somewhere durable.
Pushed to `https://github.com/Revanthkr1/fem-plate-gnn` (**private**) --
`data/raw/*.json` and `data/norm_stats.npz` are committed, so they arrive with
the clone; only `data/cache/` (gitignored, rebuilt in section 3) is missing
after this.

Because the repo is private, plain `git clone` would prompt for credentials
and fail non-interactively on Colab. The cell below reads a GitHub personal
access token from Colab's **Secrets** panel (key icon, left sidebar) instead
of hardcoding one -- add a token there first, named `GITHUB_TOKEN` (generate
one, classic, `repo` scope, at https://github.com/settings/tokens). Never
paste a real token directly into this notebook -- it gets committed to git.
(Alternative: make the repo public later once you're ready to share it, and
plain `git clone` works with no token at all.)

In [ ]:
from google.colab import userdata

# Repo is private -- add a GitHub personal access token (classic, `repo` scope)
# as a Colab secret named GITHUB_TOKEN (key icon, left sidebar) first. Never
# hardcode the token here -- this notebook file gets committed to git.
token = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/Revanthkr1/fem-plate-gnn.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies

Colab ships torch with CUDA already installed -- don't reinstall it.
`torch_geometric` installs as pure Python here too (same `torch_geometric.utils.scatter`-only
usage as AirfRANS's model.py, ported unchanged). No `airfrans` package needed --
this project doesn't depend on it.

In [ ]:
!pip install -q torch_geometric lightning pyvista pyyaml

## 3. Rebuild the cache

`data/raw/*.json` (200 cases) and `data/norm_stats.npz` came with the clone.
`data/cache/` (preprocessed graph tensors) is gitignored and cheap to rebuild --
these are small meshes (~5-6k nodes, not AirfRANS's ~180k), so this is seconds,
not the dominant cost VTU parsing was there. Safe to re-run: `preprocess_case`
skips any case already cached.

In [ ]:
import glob
import os

from src.preprocess import preprocess_split

RAW_DIR = "data/raw"
CACHE_DIR = "data/cache"

n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))
case_ids = list(range(n_cases))
preprocess_split(RAW_DIR, case_ids, CACHE_DIR)
print(f"cached {len(glob.glob(os.path.join(CACHE_DIR, 'case_*.pt')))}/{n_cases} cases")

## 4. Smoke test: one case on the real GPU (optional)

Same forward/backward timing check as the AirfRANS notebook's section 4 --
confirms the model + a real cached graph actually move to the GPU and run
before committing to a full training loop. Skip if you're just resuming a
training run.

In [ ]:
import time
import numpy as np
import torch

from src.dataset import CachedPyGPlateHoleDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

stats = dict(np.load("data/norm_stats.npz"))
ds = CachedPyGPlateHoleDataset(CACHE_DIR, [0], stats=stats)
data = ds[0].to(device)

model = MeshGraphNet(node_in_dim=3, edge_in_dim=2, out_dim=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 5. Train

Trains on 200 cases minus a 20-case validation holdout (`n_val=20`, ~10% --
see `configs/base.yaml`), reporting relative L2 per field (`u_x`, `u_y`,
`von_mises` separately -- never one blended number, per CLAUDE.md) on
validation each epoch.

**This run tests the phase-10/11 diagnostic: does more message-passing reach
fix peak-stress localization?** Phase 10 dropped the hand-engineered
`signed_distance_to_hole` feature (still gone -- `node_in_dim=3`, `[x, y,
load]` only) and, even with a clean 250-epoch run, peak-stress *location*
error got much worse (18.1mm -> 51.6mm) while magnitude error stayed about
the same (see `PROJECT_FLOW.md` phase 10). Working theory: with no distance
feature, a node's only way to learn "where's the hole" is via message-
passing hops from hole-boundary nodes, and `n_message_passing=4` (~8mm of
reach at this mesh's ~2mm spacing) is nowhere near enough for holes up to
15mm radius. `configs/base.yaml` now sets `n_message_passing=8` (~16mm
reach) to test that directly. **This is a genuine diagnostic, not a
guaranteed fix** -- even 8 hops may still fall short for the largest holes
or most distant nodes; if the location error doesn't meaningfully improve,
that tells us hop-count alone isn't the answer and something more
structural (e.g. a topology-native distance feature, or a global/virtual
node for O(1)-hop long-range context) would be needed instead.

**Starts fresh, own checkpoint subdirectory** (`DRIVE_ROOT/moremp/`) --
learned the hard way in phase 9/10 that reusing a directory (even under a
different final filename) lets `train.py`'s auto-resume silently find and
load an unrelated run's periodic checkpoint. Every new experiment gets its
own subdirectory, no exceptions.

Checkpoints go to Drive so they survive a session reset -- unlike the dataset
(cheap to re-clone + rebuild from section 1/3), a training run in progress is
not something you want to lose.

These graphs are much smaller than AirfRANS's, so OOM headroom is not the
concern `batch_size=1`/`accumulate_grad_batches=4` addressed there -- kept the
same defaults anyway since they're harmless at this scale and a larger batch
size hasn't been benchmarked yet. Doubling `n_message_passing` roughly
doubles compute per forward pass, so expect this run to take noticeably
longer than the 250-epoch phase-10 run.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/fem-plate-gnn-data"
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import glob
import yaml

from src.train import main as train_main

config = yaml.safe_load(open("configs/base.yaml"))
n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))

# Own subdirectory, not just a different filename -- see section 5 markdown.
RUN_DIR = os.path.join(DRIVE_ROOT, "moremp")
os.makedirs(RUN_DIR, exist_ok=True)

train_main(
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=os.path.join(RUN_DIR, "meshgraphnet_moremp.ckpt"),
    case_ids=list(range(n_cases)),
    model_kwargs=config["model"],
    max_epochs=config["training"]["max_epochs"],
    batch_size=config["training"]["batch_size"],
    accumulate_grad_batches=config["training"]["accumulate_grad_batches"],
    n_val=config["training"]["n_val"],
    lr=config["training"]["lr"],
    checkpoint_every_n_epochs=config["training"]["checkpoint_every_n_epochs"],
    num_workers=config["training"]["num_workers"],
    precision="16-mixed",  # GPU-specific override -- base.yaml's 32-true is for local CPU runs
)